In [8]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
df = pd.read_csv('combined_sentiment_data.csv')

Saving combined_sentiment_data.csv to combined_sentiment_data (1).csv


In [9]:
theme_keywords = {
    'Physical Strain and Exhaustion': ['physical', 'physically', 'exhausting', 'exhausted', 'legs', 'body', 'heavy', 'draining', 'tired'],
    'Productivity Pressure and Monitoring': ['break', 'overwhelming', 'rate', 'target', 'timed', 'mandatory', 'overtime pressure', 'monitored'],
    'Pay Adequacy vs Overtime Dependency': ['pay', 'salary', 'overtime', 'wage', 'compensation', 'hourly'],
    'Monotony and Isolation': ['boring', 'repetitive', 'isolated', 'isolating', 'lonely', 'monotonous', 'same thing'],
    'Favoritism and Management Fairness': ['favoritism', 'toxic', 'biased', 'unfair', 'politics']
}

In [10]:
def tag_themes(text, theme_keywords):
    matched_themes = []
    for theme, keywords in theme_keywords.items():
        if any(keyword in str(text).lower() for keyword in keywords):
            matched_themes.append(theme)
    return ', '.join(matched_themes) if matched_themes else ''

df['Theme'] = df['cleaned_text'].apply(lambda x: tag_themes(x, theme_keywords))

print(df[['cleaned_text', 'Theme']].head(15))

                      cleaned_text  \
0                           amazon   
1                             good   
2                  good experience   
3                             dhdh   
4                    good pay rate   
5        worst american sweat shop   
6                        warehouse   
7                       less ideal   
8                           amazon   
9           keep going keep trying   
10  amazon fulfillment center bdl2   
11                         procons   
12             warehouse associate   
13                           worst   
14        one full year employment   

                                                Theme  
0                                                      
1                                                      
2                                                      
3                                                      
4   Productivity Pressure and Monitoring, Pay Adeq...  
5                                                      

Initial tagging using only the summary_cleaned column returned mostly blank themes, since Glassdoor summaries are often very short (1-3 words) and don’t contain enough context to match theme keywords. Updated the approach to combine summary, review_cons, and review_pros for richer text to scan.


In [11]:
df['full_text_for_tagging'] = (
    df['cleaned_text'].fillna('') + ' ' +
    df.get('review_cons_cleaned', pd.Series(['']*len(df))).fillna('') + ' ' +
    df.get('review_pros_cleaned', pd.Series(['']*len(df))).fillna('')
)

df['Theme'] = df['full_text_for_tagging'].apply(lambda x: tag_themes(x, theme_keywords))

print(df[['full_text_for_tagging', 'Theme']].head(15))

                                full_text_for_tagging  \
0   amazon inhumane draining physically mentally i...   
1   good sticks rules completing targets otherwise...   
2   good experience limited opportunities advancem...   
3   dhdh tough body labor intensive plenty work me...   
4   good pay rate hiring permanent employees direc...   
5   worst american sweat shop management care empl...   
6   warehouse fast paced working environment weigh...   
7   less ideal people charge wasnt good decent wor...   
8   amazon management absolutely terrible communic...   
9   keep going keep trying work life less sleep wo...   
10  amazon fulfillment center bdl2 standing 10 hou...   
11  procons workplace culture bad benefits great s...   
12  warehouse associate nightshift overtime cons g...   
13  worst physical job prepare tired 4 day weeks g...   
14  one full year employment cons knowledge good r...   

                                                Theme  
0                      Physical

In [12]:
df.to_csv('combined_with_themes.csv', index=False)